In [1]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import tqdm
import time
import random
import json

# scraper_0.py

### Setup 

In [14]:
# Retrieve iFrame URL
url = "https://chicagomaroon.com/archive/"
html = requests.get(url).text

soup = BeautifulSoup(html, "html.parser")

iframe = soup.find("iframe")
iframe_src = iframe["src"]

print(iframe_src)

https://campub.lib.uchicago.edu/search/?f1-title=Daily%20Maroon


In [35]:
# Send request to iFrame URL
BASE = "https://campub.lib.uchicago.edu"

iframe_url = iframe_src if iframe_src.startswith("http") else BASE + iframe_src

html = requests.get(iframe_url).text
soup = BeautifulSoup(html, "html.parser")

In [36]:
# Extract links on default page (page 1 is not a url!!!)
# This needs to happen before pagination bc soup will change 
# (ordering is weird but necessary)

all_links = []

for doc in soup.select("div.row.docHit"):
    a_tag = doc.find("a")
    if a_tag:
        full_url = urljoin(BASE, a_tag["href"])
        all_links.append(full_url)

In [41]:
def get_pagination_links(html):
    soup = BeautifulSoup(html, "html.parser")
    
    links = []
    
    for a in soup.select('a[href*="startDoc="]'):
        href = a.get("href")
        full_url = urljoin(BASE, href)
        links.append(full_url)
    
    return list(set(links))  # remove duplicates


html = requests.get("https://campub.lib.uchicago.edu/search/?f1-title=Daily+Maroon").text
pages = get_pagination_links(html)

# for p in pages:
    # print(p)

pages = sorted(
    pages,
    key=lambda p: int(p[p.rfind('=')+1:])
)

pages = pages[1:] # Ignore first link (page 1 has no ref, will cause problems later)

print(len(pages), 'pages')

365 pages


### Crawler

In [42]:
# Functions for document url crawler

def get_doc_links_from_page(url):
    html = fetch_with_retries(url)
    
    if html is None:
        return None  # signal failure
    
    soup = BeautifulSoup(html, "html.parser")
    
    links = []
    for doc in soup.select("div.row.docHit"):
        a_tag = doc.find("a")
        if a_tag:
            full_url = urljoin(BASE, a_tag["href"])
            links.append(full_url)
    
    return links


# Exponential backoff
def fetch_with_retries(url, max_retries=5, base_delay=1):
    for attempt in range(max_retries):
        try:
            response = requests.get(url, timeout=10)
            response.raise_for_status()
            return response.text
        
        except Exception as e:
            wait = base_delay * (2 ** attempt) + random.uniform(0, 1)
            print(f"Retry {attempt+1} for {url} in {wait:.2f}s")
            time.sleep(wait)
    
    return None  # failed after retries


def save_progress(all_links, failed_pages):
    with open("links.json", "w") as f:
        json.dump(list(set(all_links)), f)
    
    with open("failed_pages.json", "w") as f:
        json.dump(failed_pages, f)


In [43]:
# Try all links, create queue with failed pages
failed_pages = []

for i, page_url in enumerate(tqdm.tqdm(pages)):
    links = get_doc_links_from_page(page_url)
    
    if links is None:
        failed_pages.append(page_url)
        continue
    
    if not links:
        print(f"Empty page (likely failure): {page_url}")
        failed_pages.append(page_url)
        print("HTML preview:", html[:500])
        continue
    
    all_links.extend(links)

    if i % 10 == 0:
        save_progress(all_links, failed_pages)
    
    if BE_POLITE:
        time.sleep(0.5)

100%|█████████████████████████████████████████| 365/365 [04:58<00:00,  1.22it/s]


TypeError: unsupported operand type(s) for &: 'list' and 'int'

In [46]:
# Retry failed pages until done

round_num = 1

while (failed_pages) and (round_num <=  1_000):
    print(f"\nRetry round {round_num}, {len(failed_pages)} pages left")
    
    new_failed = []
    
    for page_url in tqdm(failed_pages):
        links = get_doc_links_from_page(page_url)
        
        if links is None:
            new_failed.append(page_url)
            continue
        
        all_links.extend(links)
    
    failed_pages = new_failed
    round_num += 1

In [47]:
len(all_links)

7316

# scraper_1.py

### Extract pdf links 
Follows pattern (not scraped, but generated)

In [76]:
from urllib.parse import urlparse, parse_qs
import json
import tqdm

def extract_doc_id(url):
    query = urlparse(url).query
    params = parse_qs(query)
    return params.get("docId", [None])[0]

def parse_date_from_doc_id(doc_id):
    try:
        parts = doc_id.split("-")
        year = parts[2]
        md = parts[3].zfill(4)  # ensure 4 digits
        
        month = md[:2]
        day = md[2:]
        date = f"{year}-{month}-{day}"
        
        return date, year, month, day
    except Exception:
        return None

def build_doc_record(doc_url):
    doc_id = extract_doc_id(doc_url)
    
    if not doc_id:
        return None
    
    date, year, month, day = parse_date_from_doc_id(doc_id)
    
    pdf_url  = f"https://campub.lib.uchicago.edu/pdf/?docId={doc_id}"
    text_url = f"https://campub.lib.uchicago.edu/text/?docId={doc_id}"
    
    return {
        "id": doc_id,
        "date": date,  
        "year": year,
        "month": month,
        "day":day,
        "doc_url": doc_url,
        "pdf_url": pdf_url,
        "plain_text_url": text_url
    }

In [77]:
with open("output/links.json") as f:
    doc_links = json.load(f)

docs = []

for doc_url in tqdm.tqdm(doc_links):
    record = build_doc_record(doc_url)
    if record:
        docs.append(record)

with open("output/documents.json", "w") as f:
    json.dump(docs, f, indent=2)

100%|████████████████████████████████████| 7240/7240 [00:00<00:00, 53048.86it/s]


In [53]:
# import json
# from bs4 import BeautifulSoup
# import requests
# from urllib.parse import urljoin
# import time, random
# import tqdm

# with open("output/links.json") as f:
#     doc_links = json.load(f)
    
# BASE = "https://campub.lib.uchicago.edu"

# def fetch_with_retries(url, max_retries=5, base_delay=1):
#     for attempt in range(max_retries):
#         try:
#             r = requests.get(url, timeout=10)
#             r.raise_for_status()
#             return r.text
#         except Exception as e:
#             wait = base_delay * (2 ** attempt) + random.uniform(0, 1)
#             print(f"Retry {attempt+1} for {url} in {wait:.1f}s")
#             time.sleep(wait)
#     return None

# def get_pdf_link(doc_url):
#     html = fetch_with_retries(doc_url)
#     if html is None:
#         return None

#     soup = BeautifulSoup(html, "html.parser")
#     # Find the <a> with string "PDF"
#     a_tag = soup.find("a", string="PDF")
#     if a_tag:
#         return urljoin(BASE, a_tag["href"])
#     return None


# def get_doc_info(doc_url):
#     """Return a dict: title, date, plain_text_url, pdf_url"""
#     html = fetch_with_retries(doc_url)
#     if html is None:
#         return None
    
#     soup = BeautifulSoup(html, "html.parser")
    
#     # Title and Date
#     title, date = None, None
#     print(soup)
#     dl = soup.find("dl")
#     if dl:
#         dt_tags = dl.find_all("dt")
#         dd_tags = dl.find_all("dd")
#         for dt, dd in zip(dt_tags, dd_tags):
#             if dt.get_text(strip=True) == "Title:":
#                 title = dd.get_text(strip=True)
#             elif dt.get_text(strip=True) == "Date:":
#                 date = dd.get_text(strip=True)
    
#     # Plain text URL
#     plain_a = soup.find("a", string="Plain Text")
#     plain_url = urljoin(BASE, plain_a['href']) if plain_a else None
    
#     # PDF URL
#     pdf_a = soup.find("a", string="PDF")
#     pdf_url = urljoin(BASE, pdf_a['href']) if pdf_a else None
    
#     # Optional: parse date to YYYY-MM-DD for ID
#     doc_id = None
#     if date:
#         try:
#             dt = datetime.datetime.strptime(date, "%B %d, %Y")
#             doc_id = dt.strftime("%Y-%m-%d")
#         except:
#             doc_id = date  # fallback
    
#     return {
#         "id": doc_id,
#         "title": title,
#         "date": date,
#         "plain_text_url": plain_url,
#         "pdf_url": pdf_url,
#         "doc_url": doc_url
#     }

# with open("output/links.json") as f:
#     doc_links = json.load(f)

# all_docs = []
# failed_docs = []

# for doc_url in tqdm.tqdm(doc_links):
#     info = get_doc_info(doc_url)
#     if info:
#         all_docs.append(info)
#     else:
#         failed_docs.append(doc_url)
#     break

# # # Save JSON
# # with open("output/documents.json", "w") as f:
# #     json.dump(all_docs, f, indent=2)

# # with open("output/failed_docs.json", "w") as f:
# #     json.dump(failed_docs, f, indent=2)

### scraper_2.py

In [78]:
import os
import json
import requests
import time, random
from tqdm import tqdm
from bs4 import BeautifulSoup

with open("output/documents.json") as f:
    docs = json.load(f)

PDF_DIR = "output/pdfs"
TEXT_DIR = "output/plain_text"

def get_paths(doc):
    doc_id = doc["id"]
    year = doc["year"]
    month = doc["month"]
    
    pdf_path = os.path.join(PDF_DIR, year, month, f"{doc_id}.pdf")
    text_path = os.path.join(TEXT_DIR, year, month, f"{doc_id}.txt")
    
    os.makedirs(os.path.dirname(pdf_path), exist_ok=True)
    os.makedirs(os.path.dirname(text_path), exist_ok=True)
    
    return pdf_path, text_path

In [79]:
def download_pdf(url, path, max_retries=5):
    if os.path.exists(path):
        return True
    
    for attempt in range(max_retries):
        try:
            with requests.get(url, stream=True, timeout=15) as r:
                r.raise_for_status()
                
                with open(path, "wb") as f:
                    for chunk in r.iter_content(8192):
                        if chunk:
                            f.write(chunk)
            return True
        
        except Exception as e:
            wait = (2 ** attempt) + random.uniform(0, 1)
            print(f"[PDF] Retry {attempt+1} for {url} in {wait:.1f}s")
            time.sleep(wait)
    
    print("[PDF FAILED]:", url)
    return False


def download_text(url, path, max_retries=5):
    if os.path.exists(path):
        return True
    
    for attempt in range(max_retries):
        try:
            r = requests.get(url, timeout=15)
            r.raise_for_status()
            
            # Force proper encoding
            r.encoding = r.apparent_encoding
            
            # Parse HTML → clean text
            soup = BeautifulSoup(r.text, "html.parser")
            
            # Extract visible text only
            text = soup.get_text(separator="\n")
            
            # Clean extra whitespace
            lines = [line.strip() for line in text.splitlines()]
            text_clean = "\n".join(line for line in lines if line)
            
            with open(path, "w", encoding="utf-8") as f:
                f.write(text_clean)
            
            return True
        
        except Exception as e:
            wait = (2 ** attempt) + random.uniform(0, 1)
            print(f"[TEXT] Retry {attempt+1} for {url} in {wait:.1f}s")
            time.sleep(wait)
    
    print("[TEXT FAILED]:", url)
    return False

In [82]:
failed = []

for doc in tqdm(docs):
    pdf_path, text_path = get_paths(doc)
    
    # PDF
    if not download_pdf(doc["pdf_url"], pdf_path):
        failed.append({"type": "pdf", "url": doc["pdf_url"]})
    
    # TEXT
    if not download_text(doc["plain_text_url"], text_path):
        failed.append({"type": "text", "url": doc["plain_text_url"]})

  6%|███▊                                                        | 456/7240 [16:00<3:58:03,  2.11s/it]


KeyboardInterrupt: 

In [83]:
from concurrent.futures import ThreadPoolExecutor

def process_doc(doc):
    pdf_path, text_path = get_paths(doc)
    
    results = []
    
    if not download_pdf(doc["pdf_url"], pdf_path):
        results.append({"type": "pdf", "url": doc["pdf_url"]})
    
    if not download_text(doc["plain_text_url"], text_path):
        results.append({"type": "text", "url": doc["plain_text_url"]})
    
    return results

In [85]:
with ThreadPoolExecutor(max_workers=5) as executor:
    for result in tqdm(executor.map(process_doc, docs), total=len(docs)):
        failed.extend(result)

 32%|██████████████████▌                                        | 2281/7240 [37:49<4:05:41,  2.97s/it]

[PDF] Retry 1 for https://campub.lib.uchicago.edu/pdf/?docId=mvol-0004-1962-0216 in 1.8s
[TEXT] Retry 1 for https://campub.lib.uchicago.edu/text/?docId=mvol-0004-1966-1028 in 1.6s


 32%|██████████████████▌                                        | 2284/7240 [38:02<4:51:37,  3.53s/it]

[PDF] Retry 1 for https://campub.lib.uchicago.edu/pdf/?docId=mvol-0004-1914-1021 in 1.4s


 50%|████████████████████████████▍                            | 3605/7240 [1:06:13<1:06:46,  1.10s/it]


KeyboardInterrupt: 

In [ ]:
# Save failures for re-attempt
with open("failed_downloads.json", "w") as f:
    json.dump(failed, f, indent=2)